# Teste isolado — AESBE (Notícias)

Fonte candidata: **AESBE — Associação Brasileira das Empresas Estaduais
de Saneamento**, setor Saneamento. Notebook **descartável** (Fase 1) --
sem dispatcher, sem `atualizar_status_fonte`, sem gravar nada. Só valida:

1. Scraping da listagem (título, resumo, link) com deduplicação
2. Extração de texto completo + descoberta de onde a data aparece

## Confirmado antes de assumir

WordPress + Elementor confirmado (`robots.txt` simples, sem bloqueios
relevantes). Paginação `/noticias-externa/N/`, 14 posts únicos por
página.

**Sobre a duplicação (confirmado, mais estrutural do que o esperado)**:
a listagem tem DOIS mecanismos de repetição empilhados:

1. Um widget "hero" (os 2 posts mais recentes, em
   `article.elementor-post` **sem** classe `ecs-post-loop`, título em
   `h3.elementor-post__title a`) que também aparece de novo dentro da
   lista paginada de verdade (`article.elementor-post.ecs-post-loop`,
   título em `h1.elementor-heading-title a`) -- mesmo padrão do hero fixo
   já visto na ABAR.
2. **Dentro de cada `article.ecs-post-loop`**, o mesmo título/link
   aparece **3 vezes seguidas** -- não são 3 posts diferentes, são 3
   variantes responsivas do mesmo bloco Elementor
   (`elementor-hidden-desktop` / `elementor-hidden-tablet` /
   `elementor-hidden-mobile`), cada uma com seu próprio
   `h1.elementor-heading-title a` apontando pra mesma URL.

Solução: pegar só o *primeiro* `h1`/`h3` de título por `article` (resolve
a repetição interna de 3x) **e** deduplicar por URL entre todos os
`article` da página (resolve a sobreposição hero vs. lista) -- os dois
juntos, mesmo raciocínio geral já usado na ABAR.

**Sobre a data**: confirmado que não aparece na listagem -- só na página
individual, em `.elementor-post-info__item--type-date time`, como texto
puro `"DD/MM/YYYY"` (sem atributo `datetime`, diferente de ABRACE/Trata
Brasil -- aqui precisa de regex mesmo).

**Sobre o texto completo**: `.elementor-widget-theme-post-content` --
já está no `SELETORES_CONTEUDO` compartilhado (mesmo seletor do Trata
Brasil) -- não precisa de nada novo.

**Sobre o título na página individual**: tem *quatro* `<h1>` na página
(o título de verdade é o primeiro, mas os outros três são de um widget
lateral "Edição Nº ..." -- conteúdo de revista/newsletter, não
relacionado à notícia). Coincidência de ordem, não uma garantia -- por
segurança, mantém o título já certo vindo da listagem (`extrair_titulo:
None`), mesmo padrão de ABRACE/agesan_noticias/agetransp, em vez de
confiar no primeiro `<h1>` da página.

Conteúdo é institucional/advocacy do setor (propostas de política
pública, eventos, câmaras técnicas, prêmios) -- sem filtro de relevância
aqui, como pedido; isso é responsabilidade da etapa de NLP.

In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml
dbutils.library.restartPython()

In [0]:
import re
import time
import random
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests

In [0]:
# =============================================================================
# Configuração
# =============================================================================

SITE_URL = "https://aesbe.org.br/noticias-externa/"

HTTP_TIMEOUT = 30
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

PADRAO_LINHAS_VAZIAS = re.compile(r"\n{3,}")
PADRAO_DATA_AESBE = re.compile(r"(\d{2})/(\d{2})/(\d{4})")
ITENS_POR_PAGINA_ESPERADO = 14

In [0]:
def headers_aleatorios(referer: Optional[str] = None) -> dict:
    headers = {
        "User-Agent": USER_AGENT,
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",
    }
    if referer:
        headers["Referer"] = referer
    return headers


def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer=SITE_URL)
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [httpx tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None

## Teste 1 — listar a listagem (com paginação e deduplicação)

`article.elementor-post` por post -- título em `h3.elementor-post__title a`
(itens "hero") ou `h1.elementor-heading-title a` (itens `ecs-post-loop`),
pegando só o *primeiro* de cada `article` (resolve a repetição 3x
interna) e deduplicando por URL entre `article`s (resolve a sobreposição
hero vs. lista).

In [0]:
def listar_aesbe(max_paginas: int = 4) -> list[dict]:
    itens, vistos = [], set()

    for pagina in range(1, max_paginas + 1):
        url_pagina = SITE_URL if pagina == 1 else f"{SITE_URL.rstrip('/')}/{pagina}/"

        html = baixar_pagina(url_pagina)
        if not html:
            print(f"  -> falha ao baixar página {pagina}; parando.")
            break

        soup = BeautifulSoup(html, "lxml")
        artigos = soup.select("article.elementor-post")
        if not artigos:
            print(f"  -> nenhum item encontrado na página {pagina}; fim da listagem.")
            break

        novos_na_pagina = 0
        for artigo in artigos:
            tag_a = artigo.select_one("h3.elementor-post__title a") or artigo.select_one("h1.elementor-heading-title a")
            if not tag_a:
                continue
            url_item = tag_a["href"].strip()
            if url_item in vistos:
                continue
            vistos.add(url_item)
            novos_na_pagina += 1

            resumo = None
            tag_resumo = artigo.select_one(".elementor-widget-text-editor")
            if tag_resumo:
                resumo = tag_resumo.get_text(" ", strip=True)

            itens.append({
                "titulo": tag_a.get_text(strip=True),
                "url": url_item,
                "resumo": resumo,
            })

        print(f"  página {pagina}: {len(artigos)} elementos (article), {novos_na_pagina} novos (após dedup).")
        if len(artigos) < ITENS_POR_PAGINA_ESPERADO:
            break
        time.sleep(random.uniform(0.5, 1.2))

    return itens

In [0]:
itens = listar_aesbe()

print(f"\n{len(itens)} notícias listadas.\n")
for item in itens:
    print(f"- {item['titulo'][:80]}")

urls_unicas = {i["url"] for i in itens}
print(f"\nurls únicas: {len(urls_unicas)}/{len(itens)}")
print(f"\nExemplo de link: {itens[0]['url']}")
print(f"Exemplo de resumo: {itens[0]['resumo']}")

## Teste 2 — abrir uma notícia, extrair texto completo e descobrir a data

`.elementor-widget-theme-post-content` (já no `SELETORES_CONTEUDO`
compartilhado) pro texto. Data em
`.elementor-post-info__item--type-date time`, texto puro `DD/MM/YYYY`
(sem atributo `datetime` -- precisa de regex).

In [0]:
def extrair_texto_generico(html: str) -> str:
    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception:
        soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript", "iframe", "form"]):
        tag.decompose()

    base = soup.select_one(".elementor-widget-theme-post-content")
    if base is None or len(base.get_text(strip=True)) < 200:
        base = soup

    texto = base.get_text("\n", strip=True)
    return PADRAO_LINHAS_VAZIAS.sub("\n\n", texto).strip()


def extrair_data_aesbe(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    tag_data = soup.select_one(".elementor-post-info__item--type-date time")
    if not tag_data:
        return None
    m = PADRAO_DATA_AESBE.search(tag_data.get_text(strip=True))
    if not m:
        return None
    dia, mes, ano = m.groups()
    return f"{ano}-{mes}-{dia}"


def extrair_noticia_aesbe(item: dict) -> Optional[dict]:
    html = baixar_pagina(item["url"])
    if not html:
        return None

    texto = extrair_texto_generico(html)

    return {
        "titulo": item["titulo"],
        "url": item["url"],
        "published_at": extrair_data_aesbe(html),
        "texto": texto,
    }

In [0]:
AMOSTRA = 6

detalhes = []
for item in itens[:AMOSTRA]:
    print(f"\n  [item] {item['titulo'][:90]}")
    detalhe = extrair_noticia_aesbe(item)
    if detalhe is None:
        print("    -> download falhou.")
        continue
    detalhes.append(detalhe)
    print(f"    -> {len(detalhe['texto'])} chars, published_at={detalhe['published_at']}")
    time.sleep(random.uniform(0.5, 1.2))

print(f"\n{len(detalhes)}/{AMOSTRA} notícias abertas com sucesso.")
curtas = [d for d in detalhes if len(d["texto"]) < 200]
sem_data = [d for d in detalhes if not d["published_at"]]
print(f"Com texto abaixo de 200 chars: {len(curtas)}")
print(f"Sem data: {len(sem_data)}")

In [0]:
# Amostra completa da primeira notícia — pra conferir na mão se bate com o
# que aparece no site.
detalhe = detalhes[0]

print("=" * 100)
print(f"TÍTULO      : {detalhe['titulo']}")
print(f"PUBLICADO EM: {detalhe['published_at']}")
print(f"URL         : {detalhe['url']}")
print(f"TAMANHO     : {len(detalhe['texto'])} chars")
print("=" * 100)
print(detalhe["texto"][:2000])

## Conclusão da Fase 1

Os dois testes passam: listagem paginada extrai título/resumo/link
deduplicado (resolvendo tanto a repetição 3x interna quanto a
sobreposição do widget hero com a lista paginada), texto completo sai
limpo via `.elementor-widget-theme-post-content` (seletor já
compartilhado, sem precisar acrescentar nada). Confirmado o achado
esperado: **a data só aparece na página individual**
(`.elementor-post-info__item--type-date time`, texto puro, sem atributo
`datetime`).

**Avaliação para a Fase 2**: encaixa no dispatcher genérico
`ingest-scraping` -- sem Selenium, sem parsing que dependa de JS.
Precisa de uma `listar_aesbe()` própria (paginação `/N/`, seletores do
tema, dedup por URL + "primeiro título por article") e um
`extrair_data_aesbe()` próprio (regex `DD/MM/YYYY` em texto puro,
diferente do `datetime` ISO de ABRACE/Trata Brasil). Texto reaproveita
`extrair_texto_generico()` sem seletor novo. Título usa
`extrair_titulo: None` (mesmo raciocínio do H1 duplo/múltiplo já visto
em ABRACE -- aqui são 4 H1, só o primeiro é o de verdade, mas por
segurança mantém o título já certo da listagem).

Histórico médio (10 páginas / ~140 posts) -- `max_paginas` conservador
(recente, não backfill completo), mesmo critério de ANP/ABEGÁS/ABAR/Trata
Brasil/ABRACE.